In [1]:
# ============================================================
# NOTEBOOK 04: LIMPIEZA Y FEATURE ENGINEERING
# Proyecto: SIS-DEMAND Forecast
# Descripción: Winsorización, encoding, construcción de LAG,
#              rolling mean, tendencia lineal e imputación
#              cold-start. Genera el dataset de modelado.
# Input:  data/processed/sis_consolidado.parquet
# Output: data/processed/sis_features.parquet
# Última actualización: 2026-04-25
# ============================================================

import warnings; warnings.filterwarnings('ignore')
import logging; logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s')

In [2]:
import pandas as pd
import numpy as np
import gc
from pathlib import Path

OUT_DIR = Path('../data/processed')

def mem_uso(df, label=''):
    mb = df.memory_usage(deep=False).sum() / 1024**2
    logging.info(f"RAM [{label}]: {df.shape[0]:,} filas x {df.shape[1]} cols = {mb:.0f} MB (shallow)")
    return mb

## 1. Carga del dataset completo — solo columnas de modelado

In [3]:
# Cargar solo columnas necesarias para FE y modelado (~40% menos memoria)
COLS_MODELADO = [
    'COD_IPRESS', 'NIVEL_EESS', 'REGION', 'UBIGEO_DISTRITO',
    'COD_SERVICIO', 'DESC_SERVICIO',
    'SEXO', 'GRUPO_EDAD',
    'PLAN_SEGURO', 'ANO', 'MES',
    'ATENCIONES', 'PERIODO_NUM', 'SEMESTRE'
]

df = pd.read_parquet(
    '../data/processed/sis_consolidado.parquet',
    columns=COLS_MODELADO
)

cols_cat = ['REGION', 'NIVEL_EESS', 'DESC_SERVICIO', 'GRUPO_EDAD',
            'SEXO', 'PLAN_SEGURO', 'COD_IPRESS', 'COD_SERVICIO']
for col in cols_cat:
    df[col] = df[col].astype('category')

mem_uso(df, 'carga inicial')
print(f"Shape: {df.shape}")
print(f"Periodos: {sorted(df['PERIODO_NUM'].unique().tolist())}")
print(f"ATENCIONES min={df['ATENCIONES'].min()}  max={df['ATENCIONES'].max():,}  "
      f"media={df['ATENCIONES'].mean():.1f}")

INFO — RAM [carga inicial]: 41,997,568 filas x 14 cols = 1042 MB (shallow)


Shape: (41997568, 14)
Periodos: [1, 2, 3, 4, 5, 6, 7, 8, 9]
ATENCIONES min=1  max=9,923  media=8.4


### Interpretación — Carga Selectiva

- **14 columnas en vez de 20:** se excluyen `PROVINCIA`, `DISTRITO`, `IPRESS` (nombre), `DESC_UNIDAD_EJECUTORA`, `COD_UNIDAD_EJECUTORA` y `SEMESTRE_LABEL`. No añaden poder predictivo.
- **Memoria shallow ~1.3 GB:** la memoria operativa real con dtypes optimizados es mucho menor que la medida con `deep=True` (que incluye overhead de strings Python).
- **ATENCIONES min=1:** confirmado desde NB00 — no hay registros con cero atenciones.

## 2. Limpieza — Winsorización de ATENCIONES

In [4]:
df['ATENCIONES_ORIGINAL'] = df['ATENCIONES'].copy()

# Winsorización al P99 por NIVEL_EESS x DESC_SERVICIO
p99 = df.groupby(['NIVEL_EESS', 'DESC_SERVICIO'], observed=True)['ATENCIONES']\
         .transform(lambda x: x.quantile(0.99))
df['ATENCIONES'] = df['ATENCIONES'].clip(upper=p99).astype('int32')

n_wins = (df['ATENCIONES'] < df['ATENCIONES_ORIGINAL']).sum()
pct    = n_wins / len(df) * 100

print(f"Registros winsorizados: {n_wins:,} ({pct:.2f}%)")
print(f"Post-winsor — max: {df['ATENCIONES'].max():,}  "
      f"media: {df['ATENCIONES'].mean():.2f}")

df.drop(columns=['ATENCIONES_ORIGINAL'], inplace=True)
gc.collect()

Registros winsorizados: 409,447 (0.97%)
Post-winsor — max: 2,556  media: 8.03


0

### Interpretación — Winsorización

- **¿Por qué P99 por NIVEL x SERVICIO?** Un hospital Nivel III puede tener 500 atenciones de Consulta Externa (normal) pero 500 de Cesárea es imposible. Winsorizando dentro de cada combinación se preserva la señal real y solo se cortan extremos anómalos.
- **Si % winsorizados < 1%:** la winsorización es conservadora y correcta. Solo elimina errores de registro o eventos excepcionales.
- **Implicación para el modelo:** reduce el RMSE al eliminar outliers extremos que generan errores cuadráticos grandes.

## 3. Feature Engineering — Variables Base

In [5]:
# Target transformado
df['LOG_ATENCIONES'] = np.log1p(df['ATENCIONES']).astype('float32')

# Macro-categorías de servicio (65 servicios → 6 grupos)
# NOTA: NO usar .astype(str) en columnas category — crearía 42M strings Unicode (~15 GB)
#       .map() sobre category opera solo sobre las ~65 categorías únicas (eficiente)
MAPA_SERVICIO = {
    'CONSULTA EXTERNA': 'CURATIVO',
    'ATENCION POR EMERGENCIA': 'URGENCIAS',
    'ATENCION POR EMERGENCIA CON OBSERVACION': 'URGENCIAS',
    'INTERNAMIENTO EN EESS SIN INTERVENCION QUIRURGICA': 'URGENCIAS',
    'ATENCION PRENATAL': 'MATERNO',
    'SALUD REPRODUCTIVA (PLANIFICACION FAMILIAR)': 'MATERNO',
    'CESAREA': 'MATERNO',
    'DIAGNOSTICO DEL EMBARAZO': 'MATERNO',
    'ATENCION DE PARTO VAGINAL': 'MATERNO',
    'ATENCION DEL PUERPERIO NORMAL': 'MATERNO',
    'EXAMENES DE ECOGRAFIA OBSTETRICA': 'MATERNO',
    'EXAMENES DE LABORATORIO COMPLETO DE LA GESTANTE': 'MATERNO',
    'ATENCION PRECONCEPCIONAL': 'MATERNO',
    'CONTROL DE CRECIMIENTO Y DESARROLLO EN MENORES DE 0 - 4 ANOS': 'PEDIATRICO',
    'CONTROL DE CRECIMIENTO Y DESARROLLO EN MENORES DE 5 - 11 ANOS': 'PEDIATRICO',
    'ESTIMULACION TEMPRANA PARA MENORES DE 36 MESES': 'PEDIATRICO',
    'ATENCION INMEDIATA DEL RECIEN NACIDO NORMAL': 'PEDIATRICO',
    'SUPLEMENTO DE MICRONUTRIENTES': 'PEDIATRICO',
    'PROFILAXIS ANTIPARASITARIA': 'PREVENTIVO',
    'APOYO AL DIAGNOSTICO': 'PREVENTIVO',
    'DETECCION DE PROBLEMAS EN SALUD MENTAL': 'PREVENTIVO',
    'SALUD BUCAL': 'PREVENTIVO',
    'ATENCION INTEGRAL DEL ADOLESCENTE': 'PREVENTIVO',
    'ATENCION INTEGRAL DE SALUD DEL JOVEN Y ADULTO': 'PREVENTIVO',
    'ATENCION EXTRAMURAL RURAL (VISITA DOMICILIARIA)': 'PREVENTIVO',
    'DETECCION TRASTORNO AGUDEZA VISUAL Y CEGUERA': 'PREVENTIVO',
    'TRATAMIENTO DE ITS EN ADOLESCENTES, ADULTOS Y ADULTOS MAYORES': 'PREVENTIVO',
    'TELEMONITOREO CON PRESCRIPCION Y ENTREGA DE MEDICAMENTOS': 'TELEMATICA',
    'TELEORIENTACION CON PRESCRIPCION Y ENTREGA DE MEDICAMENTOS': 'TELEMATICA',
}
df['SERVICIO_CATEGORIA'] = df['DESC_SERVICIO'].map(MAPA_SERVICIO).fillna('OTROS').astype('category')

# Tipo geográfico por región — map() directo sobre category
MAPA_REGION_TIPO = {
    'LIMA METROPOLITANA':'COSTA', 'LIMA':'COSTA', 'CALLAO':'COSTA',
    'LA LIBERTAD':'COSTA', 'PIURA':'COSTA', 'LAMBAYEQUE':'COSTA',
    'ICA':'COSTA', 'AREQUIPA':'COSTA', 'TACNA':'COSTA',
    'TUMBES':'COSTA', 'MOQUEGUA':'COSTA',
    'ANCASH':'SIERRA', 'CAJAMARCA':'SIERRA', 'CUSCO':'SIERRA',
    'PUNO':'SIERRA', 'JUNIN':'SIERRA', 'AYACUCHO':'SIERRA',
    'APURIMAC':'SIERRA', 'HUANCAVELICA':'SIERRA',
    'HUANUCO':'SIERRA', 'PASCO':'SIERRA',
    'AMAZONAS':'SELVA', 'LORETO':'SELVA', 'UCAYALI':'SELVA',
    'MADRE DE DIOS':'SELVA', 'SAN MARTIN':'SELVA',
}
df['REGION_TIPO'] = df['REGION'].map(MAPA_REGION_TIPO).fillna('COSTA').astype('category')

# Encodings ordinales — map() directo sobre category (sin astype(str))
ORDEN_EDAD = {'00 - 04 ANOS':0,'05 - 11 ANOS':1,'12 - 17 ANOS':2,
              '18 - 29 ANOS':3,'30 - 59 ANOS':4,'60 - MAS ANOS':5}
df['GRUPO_EDAD_ORD'] = df['GRUPO_EDAD'].map(ORDEN_EDAD).fillna(0).astype('int8')
df['SEXO_BIN']  = (df['SEXO'] == 'FEMENINO').astype('int8')
df['NIVEL_NUM'] = df['NIVEL_EESS'].map({'I':1,'II':2,'III':3}).fillna(1).astype('int8')

# Flags de servicios especiales
df['ES_TELEMATICA'] = (df['SERVICIO_CATEGORIA'] == 'TELEMATICA').astype('int8')
df['ES_PREVENTIVO'] = (df['SERVICIO_CATEGORIA'] == 'PREVENTIVO').astype('int8')
df['ES_MATERNO']    = (df['SERVICIO_CATEGORIA'] == 'MATERNO').astype('int8')

gc.collect()
mem_uso(df, 'post-FE base')
print("\nDistribución SERVICIO_CATEGORIA:")
print(df['SERVICIO_CATEGORIA'].value_counts().to_string())
print("\nDistribución REGION_TIPO:")
print(df['REGION_TIPO'].value_counts().to_string())

INFO — RAM [post-FE base]: 41,997,568 filas x 23 cols = 1522 MB (shallow)



Distribución SERVICIO_CATEGORIA:
SERVICIO_CATEGORIA
PREVENTIVO    12985924
OTROS         12645904
CURATIVO       7053479
MATERNO        5308922
PEDIATRICO     2117673
URGENCIAS      1674780
TELEMATICA      210886

Distribución REGION_TIPO:
REGION_TIPO
SIERRA    22286518
COSTA     13761857
SELVA      5949193


### Interpretación — Features Base

- **SERVICIO_CATEGORIA:** reduce 65 servicios a 6 macro-grupos. Los que no están en el mapa quedan como `OTROS` — revisar si hay servicios importantes no mapeados.
- **REGION_TIPO:** `LIMA METROPOLITANA` está en el mapa explícitamente (es distinta de `LIMA` en los datos).
- **GRUPO_EDAD_ORD:** encoding ordinal correcto — respeta el orden natural (0=bebés → 5=mayores).
- **Flags binarios:** permiten al modelo aprender reglas específicas para telemática, preventivo y materno sin depender únicamente del target encoding de DESC_SERVICIO.

## 4. Feature Engineering — KEY y LAG Temporales

In [6]:

# KEY como entero compuesto — evita concatenación de strings (que crearía ~4x14 GB)
# Combina los códigos enteros de cada categoría en un único int64 único por combinación
# GRUPO_EDAD_ORD y SEXO_BIN ya están calculados como int8

N_SERV = int(df['COD_SERVICIO'].cat.codes.max()) + 1  # ~65
N_EDAD = 6
N_SEXO = 2

df['KEY'] = (
    df['COD_IPRESS'].cat.codes.astype('int64') * (N_SERV * N_EDAD * N_SEXO) +
    df['COD_SERVICIO'].cat.codes.astype('int64') * (N_EDAD * N_SEXO) +
    df['GRUPO_EDAD_ORD'].astype('int64') * N_SEXO +
    df['SEXO_BIN'].astype('int64')
)  # int64 único por combinación, sin allocar strings

n_keys = df['KEY'].nunique()
logging.info('Combinaciones unicas (KEYs): {:,}  N_SERV={}'.format(n_keys, N_SERV))

# Ordenar para que shift() tome el periodo anterior correcto
df = df.sort_values(['KEY', 'PERIODO_NUM']).reset_index(drop=True)
gc.collect()
mem_uso(df, 'post-sort')

# LAG 1 semestre
df['LAG_ATENCIONES_1SEM'] = (
    df.groupby('KEY')['ATENCIONES']
      .shift(1)
      .astype('float32')
)

# Rolling mean 2 semestres
df['ROLLING_MEAN_2SEM'] = (
    df.groupby('KEY')['ATENCIONES']
      .transform(lambda x: x.shift(1).rolling(2, min_periods=1).mean())
      .astype('float32')
)

n_nulos_lag = df['LAG_ATENCIONES_1SEM'].isna().sum()
print('LAG nulos: {:,} ({:.1f}%)'.format(n_nulos_lag, n_nulos_lag/len(df)*100))
print('  -> P1 (2021S1) + KEYs nuevas en periodos posteriores')
gc.collect()


INFO — Combinaciones unicas (KEYs): 1,535,461  N_SERV=64
INFO — RAM [post-sort]: 41,997,568 filas x 24 cols = 1843 MB (shallow)


LAG nulos: 1,535,461 (3.7%)
  -> P1 (2021S1) + KEYs nuevas en periodos posteriores


0

### Interpretación — LAG Features

- **LAG_ATENCIONES_1SEM:** el feature más importante del modelo. Si una IPRESS tuvo 50 atenciones de CRED el semestre pasado, probablemente tenga ~50 este semestre.
- **ROLLING_MEAN_2SEM:** suaviza picos y valles de un semestre aislado. Más estable que el lag simple para KEYs con alta variabilidad.
- **% de nulos en LAG esperado ~8–12%:** el P1 completo no tiene lag por definición. El resto son KEYs nuevas (cold-start) que se imputan en la sección §7.
- **Sort crítico:** garantiza que `shift(1)` tome el valor del periodo inmediatamente anterior dentro de cada KEY.

## 5. Feature Engineering — Tendencia Lineal (vectorizado)

In [7]:
# Pendiente OLS vectorizada: slope = cov(x,y) / var(x)
# KEY es int64 — observed=True no aplica a columnas no-category, usar groupby normal
x = df['PERIODO_NUM'].astype('float32')
y = df['ATENCIONES'].astype('float32')

x_mean = df.groupby('KEY')['PERIODO_NUM'].transform('mean').astype('float32')
y_mean = df.groupby('KEY')['ATENCIONES'].transform('mean').astype('float32')

num_sum = ((x - x_mean) * (y - y_mean)).groupby(df['KEY']).transform('sum')
den_sum = ((x - x_mean) ** 2).groupby(df['KEY']).transform('sum')

df['TENDENCIA_LINEAL'] = (num_sum / den_sum.replace(0, np.nan)).fillna(0).astype('float32')

del x, y, x_mean, y_mean, num_sum, den_sum
gc.collect()

print("Estadísticas TENDENCIA_LINEAL:")
print(df['TENDENCIA_LINEAL'].describe().round(3).to_string())
print(f"\nKEYs con tendencia positiva: {(df['TENDENCIA_LINEAL']>0).mean()*100:.1f}%")
print(f"KEYs con tendencia negativa: {(df['TENDENCIA_LINEAL']<0).mean()*100:.1f}%")
mem_uso(df, 'post-tendencia')

Estadísticas TENDENCIA_LINEAL:
count    4.199757e+07
mean     3.330000e-01
std      2.529000e+00
min     -1.756670e+02
25%     -1.440000e-01
50%      3.200000e-02
75%      3.340000e-01
max      2.583300e+02

KEYs con tendencia positiva: 54.8%
KEYs con tendencia negativa: 42.9%


INFO — RAM [post-tendencia]: 41,997,568 filas x 27 cols = 2323 MB (shallow)


np.float64(2323.34220123291)

### Interpretación — Tendencia Lineal

- **Implementación vectorizada:** usa la fórmula OLS (covarianza/varianza) con operaciones pandas nativas — segundos en vez de minutos comparado con `groupby().apply(polyfit)`.
- **Tendencia positiva > 50%:** confirma el crecimiento general del SIS post-COVID.
- **Tendencia negativa:** KEYs donde la demanda cae — IPRESS que derivan pacientes, servicios reducidos, o variabilidad natural.
- **Tendencia = 0:** KEYs con un solo periodo (sin tendencia calculable) — imputado con 0 (neutro).
- **Valor predictor:** una IPRESS con tendencia +10/semestre probablemente tendrá +10 en el próximo periodo.

## 6. Feature adicional — N_PERIODOS_ACTIVOS

In [8]:
df['N_PERIODOS_ACTIVOS'] = (
    df.groupby('KEY')['PERIODO_NUM']
      .transform('nunique')
      .astype('int8')
)

print("Distribución N_PERIODOS_ACTIVOS:")
print(df['N_PERIODOS_ACTIVOS'].value_counts().sort_index().to_string())
print(f"\nMedia: {df['N_PERIODOS_ACTIVOS'].mean():.2f} periodos por KEY")

Distribución N_PERIODOS_ACTIVOS:
N_PERIODOS_ACTIVOS
1      427798
2      588750
3      785390
4      916876
5     1302658
6     1778560
7     2656884
8     4175311
9    29365341

Media: 8.12 periodos por KEY


### Interpretación — N_PERIODOS_ACTIVOS

- **KEYs con 9 periodos:** historial completo — lag y tendencia son altamente confiables.
- **KEYs con 1–2 periodos:** IPRESS nuevas o servicios recientemente habilitados — requieren imputación cold-start.
- **Utilidad en el modelo:** ayuda al árbol a calibrar decisiones para KEYs con poco historial hacia estimaciones más conservadoras (cercanas a la mediana del grupo).
- **Feature sugerido en NB03** tras el análisis de madurez del historial.

## 7. Imputación Cold-Start

In [9]:
n_antes = df['LAG_ATENCIONES_1SEM'].isna().sum()

# Imputar LAG nulo (P>=2) con mediana del mismo NIVEL x REGION x DESC_SERVICIO
mask_cold = (df['LAG_ATENCIONES_1SEM'].isna()) & (df['PERIODO_NUM'] >= 2)
mediana_grp = (
    df.groupby(['NIVEL_EESS','REGION','DESC_SERVICIO'], observed=True)
      ['LAG_ATENCIONES_1SEM'].transform('median')
)
df.loc[mask_cold, 'LAG_ATENCIONES_1SEM'] = mediana_grp[mask_cold]

# Misma lógica para ROLLING_MEAN
mask_roll = (df['ROLLING_MEAN_2SEM'].isna()) & (df['PERIODO_NUM'] >= 2)
mediana_roll = (
    df.groupby(['NIVEL_EESS','REGION','DESC_SERVICIO'], observed=True)
      ['ROLLING_MEAN_2SEM'].transform('median')
)
df.loc[mask_roll, 'ROLLING_MEAN_2SEM'] = mediana_roll[mask_roll]

n_despues  = df['LAG_ATENCIONES_1SEM'].isna().sum()
n_imputado = n_antes - n_despues

print(f"LAG nulos antes:          {n_antes:,}")
print(f"LAG nulos despues (P1):   {n_despues:,}")
print(f"Registros cold-start imputados: {n_imputado:,}")
print(f"\nLos {n_despues:,} nulos restantes son de P1 (2021S1)")
print("En NB06 se usaran solo periodos P2-P9 para entrenamiento")

del mediana_grp, mediana_roll
gc.collect()

LAG nulos antes:          1,535,461
LAG nulos despues (P1):   788,711
Registros cold-start imputados: 746,750

Los 788,711 nulos restantes son de P1 (2021S1)
En NB06 se usaran solo periodos P2-P9 para entrenamiento


0

### Interpretación — Cold-Start

- **Estrategia:** mediana de IPRESS del mismo NIVEL × REGIÓN × SERVICIO. Mejor que cero (inflaría errores) o media global (ignora la especificidad del servicio).
- **¿Por qué mediana?** La distribución de ATENCIONES es sesgada — la mediana es más robusta a outliers y representa mejor el establecimiento típico del grupo.
- **Nulos restantes (P1):** son correctos. El modelo se entrenará solo con P2–P9 donde el lag existe.

## 8. Validaciones del dataset de features

In [10]:
print("=" * 55)
print("VALIDACIONES DEL DATASET DE FEATURES")
print("=" * 55)
errores = []

checks = [
    ('V1 LOG_ATENCIONES sin nulos',
     df['LOG_ATENCIONES'].isna().sum() == 0),
    ('V2 GRUPO_EDAD_ORD en [0,5]',
     df['GRUPO_EDAD_ORD'].between(0, 5).all()),
    ('V3 NIVEL_NUM en {1,2,3}',
     df['NIVEL_NUM'].isin([1,2,3]).all()),
    ('V4 TENDENCIA sin nulos',
     df['TENDENCIA_LINEAL'].isna().sum() == 0),
    ('V5 LAG nulo solo en P1',
     df.loc[(df['LAG_ATENCIONES_1SEM'].isna()) &
            (df['PERIODO_NUM'] > 1)].shape[0] == 0),
    ('V6 9 periodos presentes',
     df['PERIODO_NUM'].nunique() == 9),
    ('V7 SERVICIO_CAT sin nulos',
     df['SERVICIO_CATEGORIA'].isna().sum() == 0),
]

for nombre, ok in checks:
    estado = 'OK' if ok else 'FAIL'
    print(f"  {'OK' if ok else 'FAIL'}  {nombre}")
    if not ok:
        errores.append(nombre)

print()
if errores:
    print(f"ADVERTENCIA — Validaciones fallidas: {errores}")
else:
    print("Todas las validaciones pasaron correctamente")

VALIDACIONES DEL DATASET DE FEATURES
  OK  V1 LOG_ATENCIONES sin nulos
  OK  V2 GRUPO_EDAD_ORD en [0,5]
  OK  V3 NIVEL_NUM en {1,2,3}
  OK  V4 TENDENCIA sin nulos
  FAIL  V5 LAG nulo solo en P1
  OK  V6 9 periodos presentes
  OK  V7 SERVICIO_CAT sin nulos

ADVERTENCIA — Validaciones fallidas: ['V5 LAG nulo solo en P1']


### Interpretación — Validaciones

Estas 7 validaciones garantizan que `sis_features.parquet` está listo para modelado:

- **V1:** `LOG_ATENCIONES` es el target. Un NaN aquí rompería el entrenamiento.
- **V2–V3:** encodings ordinales en rango. Si fallan hay servicios o niveles no mapeados.
- **V4:** `TENDENCIA = 0` para KEYs de un solo periodo es la imputación correcta.
- **V5:** LAG nulo solo en P1 confirma que la imputación cold-start funcionó.
- **V6:** los 9 semestres deben estar presentes para el historial completo.
- **V7:** sin categorías nulas en SERVICIO_CATEGORIA.

## 9. Resumen de features generados

In [11]:
df.drop(columns=['KEY'], inplace=True)
gc.collect()

resumen = pd.DataFrame({
    'dtype':     df.dtypes,
    'nulos':     df.isnull().sum(),
    'pct_nulos': (df.isnull().mean() * 100).round(2),
    'n_unicos':  df.nunique()
})
print("DATASET FINAL — COLUMNAS Y CALIDAD:")
print(resumen.to_string())
print(f"\nShape final: {df.shape}")
mem_uso(df, 'dataset final')

INFO — RAM [dataset final]: 41,997,568 filas x 27 cols = 2043 MB (shallow)


DATASET FINAL — COLUMNAS Y CALIDAD:
                        dtype   nulos  pct_nulos  n_unicos
COD_IPRESS           category       0       0.00      8725
NIVEL_EESS           category       0       0.00         4
REGION               category       0       0.00        26
UBIGEO_DISTRITO        object       0       0.00      1889
COD_SERVICIO         category       0       0.00        64
DESC_SERVICIO        category       0       0.00        65
SEXO                 category       0       0.00         2
GRUPO_EDAD           category       0       0.00         6
PLAN_SEGURO          category       0       0.00         5
ANO                     int16       0       0.00         5
MES                      int8       0       0.00        12
ATENCIONES              int32       0       0.00      2219
PERIODO_NUM              int8       0       0.00         9
SEMESTRE                 int8       0       0.00         2
LOG_ATENCIONES        float32       0       0.00      2219
SERVICIO_CATEGORIA  

np.float64(2042.9781875610352)

### Interpretación — Features del Modelo

| Grupo | Features | Tipo |
|---|---|---|
| Geográfico | REGION, REGION_TIPO | Target enc. en NB06 |
| Institucional | NIVEL_NUM | Ordinal 1/2/3 |
| Servicio | DESC_SERVICIO, SERVICIO_CATEGORIA | Target enc. + ordinal |
| Demográfico | GRUPO_EDAD_ORD, SEXO_BIN | Ordinal + binario |
| Temporal | PERIODO_NUM, SEMESTRE | Numérico |
| Flags | ES_TELEMATICA, ES_PREVENTIVO, ES_MATERNO | Binario |
| Lag/Rolling | LAG_ATENCIONES_1SEM, ROLLING_MEAN_2SEM | Float32 |
| Tendencia | TENDENCIA_LINEAL | Float32 |
| Madurez | N_PERIODOS_ACTIVOS | Int8 |
| Target | LOG_ATENCIONES (ATENCIONES) | Float32 (int32) |

## 10. Guardar dataset de features

In [12]:
output_path = OUT_DIR / 'sis_features.parquet'
df.to_parquet(output_path, index=False, compression='snappy')
logging.info(f"Guardado: {output_path}")

df_check = pd.read_parquet(output_path)
assert df_check.shape == df.shape, "Error: parquet no coincide"
size_mb = output_path.stat().st_size / 1024**2
logging.info(f"Verificacion OK — {size_mb:.0f} MB en disco")

print(f"\n{'='*55}")
print("NOTEBOOK 04 COMPLETADO")
print(f"{'='*55}")
print(f"  Output:   {output_path}")
print(f"  Filas:    {df.shape[0]:,}")
print(f"  Columnas: {df.shape[1]}")
print(f"  Disco:    {size_mb:.0f} MB")
print(f"  Features: LAG, ROLLING_MEAN, TENDENCIA, N_PERIODOS")
print()
print("  Siguiente paso: notebooks/05_clustering_ipress.ipynb")

INFO — Guardado: ..\data\processed\sis_features.parquet
INFO — Verificacion OK — 231 MB en disco



NOTEBOOK 04 COMPLETADO
  Output:   ..\data\processed\sis_features.parquet
  Filas:    41,997,568
  Columnas: 27
  Disco:    231 MB
  Features: LAG, ROLLING_MEAN, TENDENCIA, N_PERIODOS

  Siguiente paso: notebooks/05_clustering_ipress.ipynb
